# Predicting Smartphone Addiction — Advanced XGBoost

This notebook develops the strongest independently built model for the Kaggle Playground Series S6E8 smartphone-addiction classification task.

**Metric:** ROC-AUC  
**Validation:** 5-fold stratified cross-validation  
**Best independent CV:** **0.96774 ROC-AUC**  
**Best independent Kaggle Public LB:** **0.96921 ROC-AUC**

The main improvement comes from combining:
- native-categorical XGBoost,
- leakage-safe nested cross-fitted exact-value target encoding,
- structural screen-time features,
- fold-averaged test predictions.

All scores reported here correspond to models trained from the competition data; no external prediction files are used.


## 1. Load data

The notebook was originally run on Kaggle. For local reproduction, place `train.csv`, `test.csv`, and `sample_submission.csv` under `../data/` and adjust the paths if needed.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

kaggle_dir = Path("/kaggle/input/competitions/playground-series-s6e8")
local_dir = Path("../data")

data_dir = kaggle_dir if kaggle_dir.exists() else local_dir

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

print("Data directory:", data_dir)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Submission shape:", sample_submission.shape)


## 2. Feature groups and validation design

`id` is excluded from the model. The target is `addicted_label`.  
The same stratified 5-fold split (`random_state=42`) is used throughout the experiments so that model comparisons are meaningful.


In [16]:
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

numerical_cols = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

categorical_cols = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (691369, 12)
y shape: (691369,)


In [17]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Copy data for XGBoost
X_xgb = X.copy()

# XGBoost native categorical support expects pandas "category" dtype
for col in categorical_cols:
    X_xgb[col] = X_xgb[col].astype("category")

# Same 5-fold strategy as before
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(X_xgb.dtypes)

age                         float64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours            float64
sleep_hours                 float64
notifications_per_day       float64
app_opens_per_day           float64
weekend_screen_time         float64
gender                     category
stress_level               category
academic_work_impact       category
dtype: object


## 3. Raw XGBoost baseline

XGBoost uses native categorical handling and histogram-based GPU training.  
This establishes the main tree-model baseline before target encoding or structural feature engineering.


In [18]:
xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="auc",

    tree_method="hist",
    device="cuda",
    enable_categorical=True,

    random_state=42,
    n_jobs=-1
)

In [19]:
import time

xgb_scores = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_xgb, y),
    start=1
):

    X_train_fold = X_xgb.iloc[train_idx]
    X_val_fold = X_xgb.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    xgb_model.fit(
        X_train_fold,
        y_train_fold
    )

    val_probabilities = xgb_model.predict_proba(
        X_val_fold
    )[:, 1]

    auc = roc_auc_score(
        y_val_fold,
        val_probabilities
    )

    xgb_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

elapsed_time = time.time() - start_time

print("-" * 40)
print(f"Mean CV AUC: {np.mean(xgb_scores):.6f}")
print(f"Std CV AUC:  {np.std(xgb_scores):.6f}")
print(f"Total time:   {elapsed_time / 60:.2f} minutes")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [15:13:35] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 1 AUC: 0.962936
Fold 2 AUC: 0.963626
Fold 3 AUC: 0.963786
Fold 4 AUC: 0.964469
Fold 5 AUC: 0.963313
----------------------------------------
Mean CV AUC: 0.963626
Std CV AUC:  0.000512
Total time:   0.51 minutes


### Baseline comparison

Earlier baseline experiments are kept in `02_baseline.ipynb`. Their 5-fold mean ROC-AUC scores were:

| Model | Mean CV AUC |
|---|---:|
| Logistic Regression | 0.911449 |
| CatBoost | 0.953460 |
| LightGBM | 0.960023 |
| XGBoost | 0.963626 |

The raw XGBoost model became the starting point for the advanced experiments below.


## 4. Leakage-safe exact-value target encoding

An early leave-one-out exact-value encoding experiment was rejected because the training-row encoding depended subtly on each row's own target while validation rows were encoded from a training-only mapping. That representation mismatch produced a severe validation collapse.

The corrected approach below uses **nested cross-fitting**:

1. For every outer CV training fold, create target encodings with an inner 5-fold split.
2. Each outer-training row is encoded only from other rows.
3. The outer validation fold is encoded from the full outer-training fold only.
4. Smoothing (`alpha=20`) shrinks rare exact values toward the training-fold global target mean.

This avoids direct target leakage while still capturing repeated-value signal.


In [29]:
def add_crossfit_exact_te(
    X_train,
    y_train,
    X_valid,
    columns,
    alpha=20.0,
    n_inner_splits=5,
    random_state=123
):
    X_train_new = X_train.copy()
    X_valid_new = X_valid.copy()

    inner_skf = StratifiedKFold(
        n_splits=n_inner_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in columns:

        # -----------------------------
        # OOF encoding for outer train
        # -----------------------------
        train_encoded = pd.Series(
            index=X_train.index,
            dtype="float64"
        )

        for inner_train_idx, inner_val_idx in inner_skf.split(
            X_train,
            y_train
        ):

            X_inner_train = X_train.iloc[inner_train_idx]
            X_inner_val = X_train.iloc[inner_val_idx]

            y_inner_train = y_train.iloc[inner_train_idx]

            inner_global_mean = y_inner_train.mean()

            train_key = (
                X_inner_train[col]
                .fillna(-999999.123456)
            )

            val_key = (
                X_inner_val[col]
                .fillna(-999999.123456)
            )

            stats = (
                pd.DataFrame({
                    "key": train_key.values,
                    "target": y_inner_train.values
                })
                .groupby("key")["target"]
                .agg(["sum", "count"])
            )

            smoothed_map = (
                stats["sum"] + alpha * inner_global_mean
            ) / (
                stats["count"] + alpha
            )

            encoded_values = (
                val_key
                .map(smoothed_map)
                .fillna(inner_global_mean)
            )

            train_encoded.loc[X_inner_val.index] = encoded_values.values

        # ----------------------------------
        # Full outer-train -> outer-validation
        # ----------------------------------
        outer_global_mean = y_train.mean()

        full_train_key = (
            X_train[col]
            .fillna(-999999.123456)
        )

        valid_key = (
            X_valid[col]
            .fillna(-999999.123456)
        )

        full_stats = (
            pd.DataFrame({
                "key": full_train_key.values,
                "target": y_train.values
            })
            .groupby("key")["target"]
            .agg(["sum", "count"])
        )

        full_map = (
            full_stats["sum"] + alpha * outer_global_mean
        ) / (
            full_stats["count"] + alpha
        )

        valid_encoded = (
            valid_key
            .map(full_map)
            .fillna(outer_global_mean)
        )

        X_train_new[f"{col}_exact_te"] = train_encoded
        X_valid_new[f"{col}_exact_te"] = valid_encoded.values

    return X_train_new, X_valid_new

In [30]:
import time

xgb_crossfit_te_scores = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_xgb, y),
    start=1
):

    X_train_fold = X_xgb.iloc[train_idx].copy()
    X_val_fold = X_xgb.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Proper cross-fitted exact-value target encoding
    X_train_te, X_val_te = add_crossfit_exact_te(
        X_train=X_train_fold,
        y_train=y_train_fold,
        X_valid=X_val_fold,
        columns=numerical_cols,
        alpha=20.0,
        n_inner_splits=5,
        random_state=123
    )

    xgb_model.fit(
        X_train_te,
        y_train_fold
    )

    val_probabilities = xgb_model.predict_proba(
        X_val_te
    )[:, 1]

    auc = roc_auc_score(
        y_val_fold,
        val_probabilities
    )

    xgb_crossfit_te_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

elapsed_time = time.time() - start_time

print("-" * 50)
print(f"Raw XGBoost AUC:          {np.mean(xgb_scores):.6f}")
print(f"Cross-fit Exact-TE AUC:   {np.mean(xgb_crossfit_te_scores):.6f}")
print(
    f"Improvement:              "
    f"{np.mean(xgb_crossfit_te_scores) - np.mean(xgb_scores):+.6f}"
)
print(f"Std CV AUC:               {np.std(xgb_crossfit_te_scores):.6f}")
print(f"Total time:               {elapsed_time / 60:.2f} minutes")

Fold 1 AUC: 0.966223
Fold 2 AUC: 0.966954
Fold 3 AUC: 0.967246
Fold 4 AUC: 0.967485
Fold 5 AUC: 0.966609
--------------------------------------------------
Raw XGBoost AUC:          0.963626
Cross-fit Exact-TE AUC:   0.966904
Improvement:              +0.003278
Std CV AUC:               0.000449
Total time:               0.93 minutes


**Result:** cross-fitted exact-value target encoding improved mean CV ROC-AUC from approximately **0.96363** to **0.96690**.


## 5. Structural feature engineering

The strongest feature-engineering idea was to model consistency between total daily screen time and its observed components.

The four engineered features are:

- `screen_components_observed`: number of non-missing component measurements,
- `observed_component_sum`: sum of available social/gaming/work-study screen time,
- `screen_budget_slack`: daily screen time minus the observed component sum,
- `other_screen_complete`: residual screen time when all three components are present.


In [33]:
def add_structural_features(df):
    df = df.copy()

    component_cols = [
        "social_media_hours",
        "gaming_hours",
        "work_study_hours"
    ]

    # How many screen-time components are actually observed?
    df["screen_components_observed"] = (
        df[component_cols]
        .notna()
        .sum(axis=1)
    )

    # Sum of whatever components are available
    df["observed_component_sum"] = (
        df[component_cols]
        .sum(axis=1, min_count=1)
    )

    # Remaining screen-time "budget"
    df["screen_budget_slack"] = (
        df["daily_screen_time_hours"]
        - df["observed_component_sum"]
    )

    # Strict residual only when all components exist.
    # Pandas naturally returns NaN when one component is missing.
    df["other_screen_complete"] = (
        df["daily_screen_time_hours"]
        - df["social_media_hours"]
        - df["gaming_hours"]
        - df["work_study_hours"]
    )

    return df


X_struct = add_structural_features(X_xgb)

print("Original features:", X_xgb.shape[1])
print("Structural features:", X_struct.shape[1])

print("\nNew features:")
print([
    col for col in X_struct.columns
    if col not in X_xgb.columns
])

Original features: 12
Structural features: 16

New features:
['screen_components_observed', 'observed_component_sum', 'screen_budget_slack', 'other_screen_complete']


In [34]:
import time

xgb_struct_te_scores = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_struct, y),
    start=1
):

    X_train_fold = X_struct.iloc[train_idx].copy()
    X_val_fold = X_struct.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    X_train_final, X_val_final = add_crossfit_exact_te(
        X_train=X_train_fold,
        y_train=y_train_fold,
        X_valid=X_val_fold,
        columns=numerical_cols,
        alpha=20.0,
        n_inner_splits=5,
        random_state=123
    )

    xgb_model.fit(
        X_train_final,
        y_train_fold
    )

    val_probabilities = xgb_model.predict_proba(
        X_val_final
    )[:, 1]

    auc = roc_auc_score(
        y_val_fold,
        val_probabilities
    )

    xgb_struct_te_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

elapsed_time = time.time() - start_time

print("-" * 55)
print(f"Exact-TE only AUC:       {np.mean(xgb_crossfit_te_scores):.6f}")
print(f"Structural + TE AUC:     {np.mean(xgb_struct_te_scores):.6f}")
print(
    f"Improvement:             "
    f"{np.mean(xgb_struct_te_scores) - np.mean(xgb_crossfit_te_scores):+.6f}"
)
print(f"Std CV AUC:              {np.std(xgb_struct_te_scores):.6f}")
print(f"Total time:              {elapsed_time / 60:.2f} minutes")

Fold 1 AUC: 0.967063
Fold 2 AUC: 0.967748
Fold 3 AUC: 0.968129
Fold 4 AUC: 0.968224
Fold 5 AUC: 0.967531
-------------------------------------------------------
Exact-TE only AUC:       0.966904
Structural + TE AUC:     0.967739
Improvement:             +0.000835
Std CV AUC:              0.000421
Total time:              1.05 minutes


**Result:** structural features + cross-fitted target encoding reached approximately **0.96774 mean CV ROC-AUC**, an additional improvement over target encoding alone.


## 6. Ablation study

To check whether the structural gain came from one feature or from the full set, each engineered feature was evaluated individually while keeping the same CV and target-encoding procedure.


In [35]:
structural_feature_names = [
    "screen_components_observed",
    "observed_component_sum",
    "screen_budget_slack",
    "other_screen_complete"
]

ablation_results = []

for feature_to_keep in structural_feature_names:

    X_ablation = X_xgb.copy()
    X_ablation[feature_to_keep] = X_struct[feature_to_keep]

    scores = []

    for train_idx, val_idx in skf.split(X_ablation, y):

        X_train_fold = X_ablation.iloc[train_idx].copy()
        X_val_fold = X_ablation.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        X_train_final, X_val_final = add_crossfit_exact_te(
            X_train=X_train_fold,
            y_train=y_train_fold,
            X_valid=X_val_fold,
            columns=numerical_cols,
            alpha=20.0,
            n_inner_splits=5,
            random_state=123
        )

        xgb_model.fit(
            X_train_final,
            y_train_fold
        )

        pred = xgb_model.predict_proba(
            X_val_final
        )[:, 1]

        scores.append(
            roc_auc_score(y_val_fold, pred)
        )

    ablation_results.append({
        "feature": feature_to_keep,
        "mean_auc": np.mean(scores),
        "improvement_vs_TE": (
            np.mean(scores)
            - np.mean(xgb_crossfit_te_scores)
        )
    })

ablation_df = pd.DataFrame(ablation_results)

ablation_df.sort_values(
    "mean_auc",
    ascending=False
)

,feature,mean_auc,improvement_vs_TE
2,screen_budget_slack,0.967683,0.000780
3,other_screen_complete,0.967541,0.000637
1,observed_component_sum,0.967083,0.000180
0,screen_components_observed,0.966929,0.000026


The largest individual gains came from `screen_budget_slack` and `other_screen_complete`, while the full four-feature set performed best overall.

This ablation step was useful because it showed that the improvement was not simply caused by increasing dimensionality.


## 7. Final model and test prediction

The final independently developed solution uses:

- the original 12 predictors,
- four structural features,
- nested cross-fitted exact-value target encoding of the numerical columns,
- the original XGBoost configuration,
- 5-fold OOF validation,
- mean and rank-averaged test predictions.

The final OOF score reproduced the development result at **0.967738 ROC-AUC**.


In [50]:
from scipy.stats import rankdata
import time

# =========================================================
# Prepare TEST
# =========================================================

X_test = test.drop(columns=["id"]).copy()

# Match categorical dtype/categories with training data
for col in categorical_cols:
    X_test[col] = pd.Categorical(
        X_test[col],
        categories=X_xgb[col].cat.categories
    )

# Same structural features used by our champion
X_test_struct = add_structural_features(X_test)

# =========================================================
# 5-FOLD OOF + TEST ENSEMBLE
# =========================================================

oof_pred = np.zeros(len(X_struct))
test_fold_preds = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_struct, y),
    start=1
):

    print(f"\n========== Fold {fold} ==========")

    X_train_fold = X_struct.iloc[train_idx].copy()
    X_val_fold = X_struct.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Combine validation + test so TE is built only once
    n_val = len(X_val_fold)

    X_eval_combined = pd.concat(
        [
            X_val_fold.reset_index(drop=True),
            X_test_struct.reset_index(drop=True)
        ],
        axis=0,
        ignore_index=True
    )

    # Leakage-safe exact-value TE
    X_train_final, X_eval_final = add_crossfit_exact_te(
        X_train=X_train_fold,
        y_train=y_train_fold,
        X_valid=X_eval_combined,
        columns=numerical_cols,
        alpha=20.0,
        n_inner_splits=5,
        random_state=123
    )

    # Split validation and test again
    X_val_final = X_eval_final.iloc[:n_val]
    X_test_final = X_eval_final.iloc[n_val:]

    # Fresh champion XGBoost
    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,

        objective="binary:logistic",
        eval_metric="auc",

        tree_method="hist",
        device="cuda",
        enable_categorical=True,

        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_final,
        y_train_fold
    )

    # Validation
    val_pred = model.predict_proba(
        X_val_final
    )[:, 1]

    oof_pred[val_idx] = val_pred

    fold_auc = roc_auc_score(
        y_val_fold,
        val_pred
    )

    print(f"Fold {fold} AUC: {fold_auc:.6f}")

    # Test
    test_pred = model.predict_proba(
        X_test_final
    )[:, 1]

    test_fold_preds.append(test_pred)


# =========================================================
# OOF SCORE
# =========================================================

overall_auc = roc_auc_score(
    y,
    oof_pred
)

print("\n" + "=" * 55)
print(f"OOF AUC: {overall_auc:.6f}")


# =========================================================
# SUBMISSION 1 — probability averaging
# =========================================================

test_pred_mean = np.mean(
    test_fold_preds,
    axis=0
)

submission_mean = sample_submission.copy()
submission_mean["addicted_label"] = test_pred_mean

submission_mean.to_csv(
    "/kaggle/working/submission_xgb_struct_te_mean.csv",
    index=False
)


# =========================================================
# SUBMISSION 2 — rank averaging
# =========================================================

rank_predictions = []

for pred in test_fold_preds:
    rank_predictions.append(
        rankdata(pred) / len(pred)
    )

test_pred_rank = np.mean(
    rank_predictions,
    axis=0
)

submission_rank = sample_submission.copy()
submission_rank["addicted_label"] = test_pred_rank

submission_rank.to_csv(
    "/kaggle/working/submission_xgb_struct_te_rank.csv",
    index=False
)


elapsed = time.time() - start_time

print(f"Time: {elapsed / 60:.2f} minutes")

print("\nSaved:")
print("/kaggle/working/submission_xgb_struct_te_mean.csv")
print("/kaggle/working/submission_xgb_struct_te_rank.csv")


========== Fold 1 ==========
Fold 1 AUC: 0.967063

========== Fold 2 ==========
Fold 2 AUC: 0.967748

========== Fold 3 ==========
Fold 3 AUC: 0.968129

========== Fold 4 ==========
Fold 4 AUC: 0.968224

========== Fold 5 ==========
Fold 5 AUC: 0.967531

OOF AUC: 0.967738
Time: 1.15 minutes

Saved:
/kaggle/working/submission_xgb_struct_te_mean.csv
/kaggle/working/submission_xgb_struct_te_rank.csv


## 8. Final result

| Result | ROC-AUC |
|---|---:|
| Logistic Regression baseline | 0.91145 |
| LightGBM baseline | 0.96002 |
| Raw XGBoost | 0.96363 |
| XGBoost + cross-fitted exact-value TE | 0.96690 |
| **XGBoost + structural features + cross-fitted TE** | **0.96774 CV** |
| **Independent Kaggle Public LB** | **0.96921** |

### What did not help

Several experiments were tested and rejected rather than added to the final model:

- generic ratio/difference feature engineering: essentially neutral,
- heavier XGBoost regularization: worse,
- periodic sine/cosine transforms: slightly worse,
- decimal-precision features: negligible improvement,
- RealMLP experiments: too computationally expensive for the available competition time and not used in the final submission.

### Key takeaway

The largest improvement did not come from a more complex model. It came from **validation-aware representation engineering**: repeated exact numerical values carried useful signal, but exploiting that signal safely required nested cross-fitting to prevent target leakage.
